In [ ]:
include("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/config/requirements.jl");
include("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/utils/FFT_perso.jl");

In [ ]:
#ω_a = 28.25396662345805
ω_a = 25.338776456203686

In [ ]:
const ∞ = Inf
ħ=1
e=1
β = ∞ # Case zero temperature T=0, β → ∞
ϕ_quantum=1/2

#### Caracterization of the circuit #####
#ω_a = 3.97124*(rand(1)[1]/5+0.9)*2*pi
ω_b = 2*ω_a

φ_a = 0.11
Z_a = 2*ϕ_quantum^2*φ_a^2
C_a_inv = Z_a*ω_a
E_Ca = C_a_inv/2

φ_b = 0.204
Z_b = 2*ϕ_quantum^2*φ_b^2
C_b_inv = Z_b*ω_b
E_Cb = C_b_inv/2

E_J = 37.12*2*pi*1.5


##Drive
ω_d=ω_b
ω_p=2ω_a-ω_d

ϵ_0 = 0.95
g2 = besselj(1,ϵ_0)*E_J*φ_a^2*φ_b # (Leon's paper)
ϵ_d=10.12*g2

buffer_phase = 0.0
pump_phase = 0.0;

##Env

κ_b=1/10.4; # 1/ns from Leon's paper
κ_2 = 4*g2^2/κ_b
κ_a = 0;#3e-3*κ_2 #from Leon's paper

In [ ]:
κ_a_list = 10.0 .^range(start = -3, stop = -0.5, length = 16*2+8)


t_max = 130/κ_b
t_min = 0.0
t_min_save = t_max - 10/κ_b
tspan = (0.0, t_max) # Time span for the simulation
dt = 1 / (50 * ω_a) # Time step for the simulation

dt_save=2pi/(ω_b)
t_save = max(t_min_save,0):dt_save:t_max


b_0=0.0
environment = MeanField(κ_a, κ_b)

ϵ_0 = 0.62
g2 = besselj(1,ϵ_0)*E_J*φ_a^2*φ_b # (Leon's paper)
ϵ_d = 10.12*g2
drive = DriveTwoModesBasics(ϵ_d, ω_d, buffer_phase, ϵ_0, ω_p, pump_phase)
drive_prepared = prepare(drive)

t_list = collect(range(max(t_min_save,0),step = dt_save,stop = t_max));

In [ ]:
Im_a_list=range(-4,4,10)
Re_a_list=range(-5,5,10)
Final_all_a = Dict{Float64,Dict{Tuple{Float64,Float64}, Vector{ComplexF32}}}()
Final_all_b = Dict{Float64,Dict{Tuple{Float64,Float64}, Vector{ComplexF32}}}()

Threads.@threads for κ_a in κ_a_list
    environment = MeanField(κ_a, κ_b)
    Final_mode_a = Dict{Tuple{Float64,Float64}, Vector{ComplexF32}}()
    Final_mode_b = Dict{Tuple{Float64,Float64}, Vector{ComplexF32}}()
    for i in 1:length(Re_a_list)
        re_i = Re_a_list[i]
        for j in 1:length(Im_a_list)
            im_j = Im_a_list[j]
            a_0_j = re_i + 1im*im_j

            system = SystemATS_ladder(ω_a, ω_b, E_J, E_Ca, E_Cb, a_0_j, b_0)
            system_prepared = prepare(system)
            environment_prepared = prepare(system,environment);

            u_0 = get_init(system_prepared, environment_prepared)
            p = SimulationParams(system_prepared, drive_prepared, environment_prepared)

            a, b, t_list = default_solver(u_0, (t_min, t_max), p, dt, t_save = t_save, equation = default_equation!)

            Final_mode_a[(re_i,im_j)] = a
            Final_mode_b[(re_i,im_j)] = b
        end
    end
    Final_all_a[κ_a] = Final_mode_a
    Final_all_b[κ_a] = Final_mode_b
end

In [ ]:
for κ_a in κ_a_list

    Final_mode = Final_all_a[κ_a]
    p0 = plot()

    # Overlay scatter points
    for (re_i, im_j) in keys(Final_mode)
        z = Final_mode[(re_i, im_j)]
        idx_even = 2:2:length(z)
        idx_odd  = 1:2:length(z)
        scatter!(
            p0,
            real.(z[idx_even]),
            imag.(z[idx_even]),
            ms = 1,
            color = :blue,
            label = "",
        )

        scatter!(
            p0,
            real.(z[idx_odd]),
            imag.(z[idx_odd]),
            ms = 1,
            color = :red,
            label = "",
        )
    end

    # Add labels and title. 
    plot!(p0,
          xlabel=L"\Re(a)",
          ylabel=L"\Im(a)",
          title=L"g_2/2\pi="*string(round(g2/2pi*1000,digits=3))*" MHz",
          top_margin=5mm)

    display(p0)
end


In [ ]:
t_max = 130/κ_b
t_min = 0.0
t_min_save = t_max - 10/κ_b
tspan = (0.0, t_max) # Time span for the simulation
dt = 1 / (50 * ω_a) # Time step for the simulation

dt_save=2pi/(ω_a)/7/sqrt(2)
t_save = max(t_min_save,0):dt_save:t_max


b_0=0.0
environment = MeanField(κ_a, κ_b)

ϵ_0 = 0.58
g2 = besselj(1,ϵ_0)*E_J*φ_a^2*φ_b # (Leon's paper)
ϵ_d = 10.12*g2
drive = DriveTwoModesBasics(ϵ_d, ω_d, buffer_phase, ϵ_0, ω_p, pump_phase)
drive_prepared = prepare(drive)

t_list = collect(range(max(t_min_save,0),step = dt_save,stop = t_max));

Im_a_list=range(-8,8,10)
Re_a_list=range(-10,10,10)

κ_a = κ_b/2
environment = MeanField(κ_a, κ_b)
Final_mode_a = Dict{Tuple{Float64,Float64}, Vector{ComplexF32}}()
Final_mode_b = Dict{Tuple{Float64,Float64}, Vector{ComplexF32}}()
for i in 1:length(Re_a_list)
    re_i = Re_a_list[i]
    for j in 1:length(Im_a_list)
        im_j = Im_a_list[j]
        a_0_j = re_i + 1im*im_j

        system = SystemATS_ladder(ω_a, ω_b, E_J, E_Ca, E_Cb, a_0_j, b_0)
        system_prepared = prepare(system)
        environment_prepared = prepare(system,environment);

        u_0 = get_init(system_prepared, environment_prepared)
        p = SimulationParams(system_prepared, drive_prepared, environment_prepared)

        a, b, t_list = default_solver(u_0, (t_min, t_max), p, dt, t_save = t_save, equation = default_equation!)
        x_a, p_a = (a+conj(a))/sqrt(2), 1im*(conj(a)-a)/sqrt(2)
        x_b, p_b = (b+conj(b))/sqrt(2), 1im*(conj(b)-b)/sqrt(2)

        # First change of variables
        ϵ_p = ϵ_0
        Q_a, Q_b = 1/κ_a, 1/κ_b
        Q_a_tilde, Q_b_tilde = ω_a/κ_a, ω_b/κ_b
        β_a, β_b = 4E_J*sin(ϵ_p)φ_a^2/(ω_a), 4E_J*sin(ϵ_p)φ_b^2/ω_b
        λ_a, λ_b = φ_a, φ_b
        ϵ_λ_d = ϵ_d*2*λ_b/ω_b
        ν_d = 1;

        # Second change of variables
        κ_a_tilde, κ_b_tilde = 1/(4*Q_a_tilde*sqrt(1-1/(4Q_b_tilde^2))), 1/(2*Q_b_tilde*sqrt(1-1/(4Q_b_tilde^2)))
        β_a_tilde, β_b_tilde = β_a/(2-1/(2Q_b_tilde^2)), β_b/(1-1/(4Q_b_tilde^2))
        ϵ_d_tilde = ϵ_λ_d/(1-1/(4Q_b_tilde^2))
        ν_d_tilde = ν_d/sqrt(1-1/(4Q_b_tilde^2))
        γ = (1-1/(4Q_a_tilde^2))/(1-1/(4Q_b_tilde^2))/2;


        x_lam_a, x_lam_b, p_lam_a, p_lam_b = sqrt(2)*λ_a*x_a, sqrt(2)*λ_b*x_b, sqrt(2)*λ_a*p_a, sqrt(2)*λ_b*p_b
        x_a_tilde, x_b_tilde, p_a_tilde, p_b_tilde = x_lam_a, x_lam_b, (p_lam_a+x_lam_a/(2*Q_b_tilde))/sqrt(1-1/(4*Q_b_tilde^2)), (p_lam_b+x_lam_b/(2*Q_b_tilde))/sqrt(1-1/(4*Q_b_tilde^2))
        s = sqrt(1-1/(4*Q_b_tilde^2))*t_list
        u_a =@. cos(ω_b*t_list/2)*x_a_tilde - sin(ω_b*t_list/2)*p_a_tilde
        v_a =@. sin(ω_b*t_list/2)*x_a_tilde + cos(ω_b*t_list/2)*p_a_tilde
        u_b =@. cos(ω_b*t_list)*x_b_tilde - sin(ω_b*t_list)*p_b_tilde
        v_b =@. sin(ω_b*t_list)*x_b_tilde + cos(ω_b*t_list)*p_b_tilde;

        a_R = (u_a+1im*v_a)/sqrt(2)
        b_R = (u_b+1im*v_b)/sqrt(2)
        Final_mode_a[(re_i,im_j)] = a_R
        Final_mode_b[(re_i,im_j)] = b_R
    end
end

In [ ]:
p0 = plot()

# Overlay scatter points
for (re_i, im_j) in keys(Final_mode_a)
    z = Final_mode_a[(re_i, im_j)]
    scatter!(
        p0,
        real.(z),
        imag.(z),
        ms = 1,
        color = :red,
        label = "",
    )
end

# Add labels and title. 
plot!(p0,
        xlabel=L"\Re(a)",
        ylabel=L"\Im(a)",
        title=L"g_2/2\pi="*string(round(g2/2pi*1000,digits=3))*" MHz",
        top_margin=5mm)

In [ ]:
p0 = plot()

# Overlay scatter points
for (re_i, im_j) in keys(Final_mode_b)
    z = Final_mode_b[(re_i, im_j)]
    scatter!(
        p0,
        real.(z),
        imag.(z),
        ms = 1,
        color = :red,
        label = "",
    )
end

# Add labels and title. 
plot!(p0,
        xlabel=L"\Re(b)",
        ylabel=L"\Im(b)",
        title=L"g_2/2\pi="*string(round(g2/2pi*1000,digits=3))*" MHz",
        top_margin=5mm)

# Logistic map graph

In [ ]:
kappa_a_list = κ_a_list = 10.0 .^range(start = -1.4, stop = -1.1, length = 16*4)


t_max = 130/κ_b
t_min = 0.0
t_min_save = t_max - 10/κ_b
tspan = (0.0, t_max) # Time span for the simulation
dt = 1 / (50 * ω_a) # Time step for the simulation

dt_save=2pi/(ω_a)
t_save = max(t_min_save,0):dt_save:t_max


b_0=0.0
environment = MeanField(κ_a, κ_b)

ϵ_0 = 0.62
g2 = besselj(1,ϵ_0)*E_J*φ_a^2*φ_b # (Leon's paper)
ϵ_d = 10.12*g2
drive = DriveTwoModesBasics(ϵ_d, ω_d, buffer_phase, ϵ_0, ω_p, pump_phase)
drive_prepared = prepare(drive)

t_list = collect(range(max(t_min_save,0),step = dt_save,stop = t_max));

In [ ]:
Im_a_list=range(-4,4,10)
Re_a_list=range(-5,5,25)
Final_all_a = Vector{Vector{ComplexF32}}(undef, length(κ_a_list))
#Final_all_b = Vector{Vector{ComplexF32}}(undef, length(κ_a_list))

Threads.@threads for idx in eachindex(κ_a_list)
    κ_a = κ_a_list[idx]
    environment = MeanField(κ_a, κ_b)
    Final_mode_a = Vector{ComplexF32}()
    #Final_mode_b = Vector{ComplexF32}()
    for i in 1:length(Re_a_list)
        re_i = Re_a_list[i]
        for j in 1:length(Im_a_list)
            im_j = Im_a_list[j]
            a_0_j = re_i + 1im*im_j

            system = SystemATS_ladder(ω_a, ω_b_0, E_J, E_Ca, E_Cb, a_0_j, b_0)
            system_prepared = prepare(system)
            environment_prepared = prepare(system,environment);

            u_0 = get_init(system_prepared, environment_prepared)
            p = SimulationParams(system_prepared, drive_prepared, environment_prepared)

            a, _, t_list = default_solver(u_0, (t_min, t_max), p, dt, t_save = t_save, equation = default_equation!)

            mask  = (real.(a) .>2) .& (imag.(a) .>.5)

            append!(Final_mode_a, a[mask])
            #append!(Final_mode_b, b[mask])
        end
    end
    Final_all_a[idx] = Final_mode_a
end

In [ ]:
x_all = Float64[]
y_all = Float64[]
z_all = Float64[]

for i in eachindex(Final_all_a)
    v = Final_all_a[i]
    append!(x_all, real.(v))
    append!(y_all, imag.(v))
    append!(z_all, fill(κ_a_list[i], length(v)))
end

In [ ]:
using JSON
data = Dict(
    "x_all" => x_all,
    "y_all" => y_all,
    "z_all" => z_all,
)
open("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/analysis/ouput/data_logistic.json", "w") do io
    JSON.print(io, data)
end

# Period doubling comparison

In [ ]:
κ_a = 0.1

t_max = 130/κ_b
t_min = 0.0
t_min_save = 30/κ_b#t_max - 30/κ_b
tspan = (0.0, t_max) # Time span for the simulation
dt = 1 / (50 * ω_a) # Time step for the simulation

dt_save=1/(50ω_a)
t_save = t_min_save:dt_save:t_max

a_0 = -2-2im
b_0=0.0


ϵ_0 = 0.62
g2 = besselj(1,ϵ_0)*E_J*φ_a^2*φ_b # (Leon's paper)
ϵ_d = 10.12*g2
drive = DriveTwoModesBasics(ϵ_d, ω_d, buffer_phase, ϵ_0, ω_p, pump_phase)
drive_prepared = prepare(drive)

t_list = collect(range(max(t_min_save,0),step = dt_save,stop = t_max));
environment = MeanField(κ_a, κ_b)
system = SystemATS_ladder(ω_a, ω_b_0, E_J, E_Ca, E_Cb, a_0, b_0)
system_prepared = prepare(system)
environment_prepared = prepare(system,environment);

u_0 = get_init(system_prepared, environment_prepared)
p = SimulationParams(system_prepared, drive_prepared, environment_prepared)
prob = ODEProblem(default_equation!, u_0, tspan, p)

sol = solve(prob, Tsit5(),dt=dt,
    saveat = t_save
    )
#sol = solve(prob, Tsit5(); reltol=1e-8, abstol=1e-10,
#    saveat = t_save
#    )

a = getindex.(sol.u,1)
b = getindex.(sol.u,2)

t_list = sol.t;

In [ ]:
plot(real.(a), imag.(a), label="", zcolor=t_list)
plot!(xlabel=L"\Re(a)", ylabel=L"\Im(a)")

In [ ]:
Re_a = real.(a)
F, freqs = FFT_perso(Re_a.-sum(Re_a)/length(Re_a), t_list)
plot(freqs, abs.(F), linewidth=1, label="")
plot!(xlabel="Frequency (GHz)", ylabel="|FFT|",
xlims=(0,20), ylims=(1,1e6),
yaxis=:log)

In [ ]:
κ_a = 0.0517947467923121#0.1

t_max = 130/κ_b
t_min = 0.0
t_min_save = 30/κ_b #t_max - 30/κ_b
tspan = (0.0, t_max) # Time span for the simulation
dt = 1 / (50 * ω_a) # Time step for the simulation

dt_save=1/(50ω_a)
t_save = t_min_save:dt_save:t_max

a_0 = -2-2im
b_0=0.0


ϵ_0 = 0.62
g2 = besselj(1,ϵ_0)*E_J*φ_a^2*φ_b # (Leon's paper)
ϵ_d = 10.12*g2
drive = DriveTwoModesBasics(ϵ_d, ω_d, buffer_phase, ϵ_0, ω_p, pump_phase)
drive_prepared = prepare(drive)

t_list = collect(range(max(t_min_save,0),step = dt_save,stop = t_max));
environment = MeanField(κ_a, κ_b)
system = SystemATS_ladder(ω_a, ω_b_0, E_J, E_Ca, E_Cb, a_0, b_0)
system_prepared = prepare(system)
environment_prepared = prepare(system,environment);

u_0 = get_init(system_prepared, environment_prepared)
p = SimulationParams(system_prepared, drive_prepared, environment_prepared)
prob = ODEProblem(default_equation!, u_0, tspan, p)

sol = solve(prob, Tsit5(),dt=dt,
    saveat = t_save
    )
#sol = solve(prob, Tsit5(); reltol=1e-8, abstol=1e-10,
#    saveat = t_save
#    )

a_2 = getindex.(sol.u,1)
b = getindex.(sol.u,2)

t_list = sol.t;

In [ ]:
scatter(real.(a), imag.(a), label="", zcolor=t_list)
plot!(xlabel=L"\Re(a)", ylabel=L"\Im(a)")

In [ ]:
Re_a_2 = real.(a_2)
F_2, freqs_2 = FFT_perso(Re_a_2.-sum(Re_a_2)/length(Re_a_2), t_list)
plot(freqs_2, abs.(F_2), linewidth=1, label=L"\kappa_a=0.0518"*" GHz") 
plot!(freqs, abs.(F), linewidth=1, label=L"\kappa_a=0.1"*" GHz")
plot!(xlabel="Frequency (GHz)", ylabel="|FFT|",
xlims=(0,20), ylims=(1,1e6),
yaxis=:log)